# Energy-Based Pseudo-Labeling for YouTube Videos

**Strategy:**
1. Extract utterance-level features from 620 audio files
2. Use **RMS energy** as proxy for laughter (high energy = likely laughter)
3. Label utterances with energy > threshold as positive
4. Train F0 model on pseudo-labels
5. Filter by confidence

**Why this works:**
- Laughter has distinct energy bursts
- Simple, fast, no external models needed
- Works on any comedy video

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive')

!pip install -q librosa numpy pandas scikit-learn

import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
import json
import os

BASE = '/content/drive/MyDrive/chuckle_net'
AUDIO_DIR = f'{BASE}/audio'
OUTPUT_DIR = f'{BASE}/energy_labeled'
os.makedirs(OUTPUT_DIR, exist_ok=True)

audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a')]
print(f'Found {len(audio_files)} audio files')

In [ ]:
# === FAST FEATURE EXTRACTION (NO PYIN) ===
def extract_utterance_features_fast(audio_path, sr=22050):
    """Fast extraction using RMS energy only (no pyin)."""
    try:
        # Load audio
        y, _ = librosa.load(audio_path, sr=sr, mono=True)
        
        # Get RMS energy
        hop_length = 512
        rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
        times = librosa.times_like(rms, sr=sr, hop_length=hop_length)
        
        # Find silence boundaries
        silence_mask = rms < 0.01
        
        utterances = []
        in_utt = False
        start_idx = 0
        
        for i, (t, is_sil) in enumerate(zip(times, silence_mask)):
            if not is_sil and not in_utt:
                start_idx = i
                in_utt = True
            elif is_sil and in_utt:
                end_idx = i
                start_t = times[start_idx]
                end_t = times[end_idx]
                
                if end_t - start_t > 0.3:  # Min 0.3s
                    rms_vals = rms[start_idx:end_idx]
                    
                    # Load segment for feature extraction
                    start_sample = int(start_t * sr)
                    end_sample = int(end_t * sr)
                    segment = y[start_sample:end_sample]
                    
                    if len(segment) < 1024:
                        in_utt = False
                        continue
                    
                    # FAST features (no pyin!)
                    feat = []
                    
                    # Energy
                    feat.extend([
                        np.mean(rms_vals),
                        np.max(rms_vals),
                        np.std(rms_vals),
                        np.max(rms_vals) / (np.mean(rms_vals) + 1e-8)
                    ])
                    
                    # ZCR
                    zcr = librosa.feature.zero_crossing_rate(segment)[0]
                    feat.extend([np.mean(zcr), np.std(zcr)])
                    
                    # MFCCs (fast!)
                    mfccs = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13)
                    for i in range(13):
                        feat.extend([np.mean(mfccs[i]), np.std(mfccs[i])])
                    
                    # Spectral (fast!)
                    spec_cent = librosa.feature.spectral_centroid(y=segment, sr=sr)[0]
                    spec_bw = librosa.feature.spectral_bandwidth(y=segment, sr=sr)[0]
                    spec_rolloff = librosa.feature.spectral_rolloff(y=segment, sr=sr)[0]
                    feat.extend([np.mean(spec_cent), np.std(spec_cent)])
                    feat.extend([np.mean(spec_bw), np.std(spec_bw)])
                    feat.extend([np.mean(spec_rolloff), np.std(spec_rolloff)])
                    
                    # Spectral contrast
                    spec_contrast = librosa.feature.spectral_contrast(y=segment, sr=sr)
                    feat.extend([np.mean(spec_contrast), np.std(spec_contrast)])
                    
                    # Label: high energy = laughter
                    label = 1 if np.max(rms_vals) > 0.15 else 0
                    
                    utterances.append({
                        'start': start_t,
                        'end': end_t,
                        'duration': end_t - start_t,
                        'rms_max': float(np.max(rms_vals)),
                        'features': np.array(feat, dtype=np.float32),
                        'label': label
                    })
                
                in_utt = False
        
        return utterances
        
    except Exception as e:
        return []

print('Function defined')

In [ ]:
# === PROCESS ALL AUDIO FILES ===
all_samples = []
stats = {'files': 0, 'utterances': 0, 'positive': 0}

for audio_file in tqdm(audio_files, desc='Processing'):
    video_id = audio_file.replace('.m4a', '')
    audio_path = f'{AUDIO_DIR}/{audio_file}'
    
    samples = extract_utterance_features_fast(audio_path)
    
    for s in samples:
        s['video_id'] = video_id
        all_samples.append(s)
    
    stats['files'] += 1
    stats['utterances'] += len(samples)
    stats['positive'] += sum(s['label'] for s in samples)

print(f'\nProcessed {stats["files"]} files')
print(f'Total utterances: {stats["utterances"]}')
print(f'Positive (high energy): {stats["positive"]} ({100*stats["positive"]/max(1,stats["utterances"]):.1f}%)')

In [ ]:
# === ANALYZE ENERGY DISTRIBUTION ===
rms_values = [s['rms_max'] for s in all_samples]
print(f'RMS max: min={min(rms_values):.4f}, max={max(rms_values):.4f}, mean={np.mean(rms_values):.4f}')

# Try different thresholds
print('\nThreshold analysis:')
for thresh in [0.05, 0.1, 0.15, 0.2, 0.3]:
    n_pos = sum(1 for s in all_samples if s['rms_max'] > thresh)
    print(f'  {thresh}: {n_pos} positive ({100*n_pos/len(all_samples):.1f}%)')

In [ ]:
# === CREATE DATASET ===
# Use threshold that gives ~15-20% positive
THRESHOLD = 0.15

X_list = []
y_list = []
video_ids = []

for s in all_samples:
    s['label'] = 1 if s['rms_max'] > THRESHOLD else 0
    X_list.append(s['features'])
    y_list.append(s['label'])
    video_ids.append(s['video_id'])

X = np.array(X_list)
y = np.array(y_list)

print(f'Dataset: {len(X)} samples')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

# Save
np.savez(f'{OUTPUT_DIR}/energy_labeled_data.npz', 
          X=X, y=y, video_ids=np.array(video_ids))
print(f'Saved to {OUTPUT_DIR}/energy_labeled_data.npz')

In [ ]:
# === TRAIN MODEL ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score

# Video-level split
unique_videos = list(set(video_ids))
train_vids, test_vids = train_test_split(unique_videos, test_size=0.2, random_state=42)

train_mask = np.array([v in train_vids for v in video_ids])
test_mask = np.array([v in test_vids for v in video_ids])

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f'Train: {len(X_train)} samples, pos={y_train.sum()} ({100*y_train.mean():.1f}%)')
print(f'Test: {len(X_test)} samples, pos={y_test.sum()} ({100*y_test.mean():.1f}%)')

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_scaled, y_train)

# Evaluate
y_pred = clf.predict(X_test_scaled)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)

print(f'\n=== RESULTS ===')
print(f'F1: {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Reacall: {rec:.4f}')

In [ ]:
# === TRY DIFFERENT THRESHOLDS ===
results = []
for thresh in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]:
    y_thresh = (np.array([s['rms_max'] for s in all_samples]) > thresh).astype(int)
    
    y_train_t = y_thresh[train_mask]
    y_test_t = y_thresh[test_mask]
    
    pos_rate = y_thresh.mean()
    
    clf_t = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf_t.fit(X_train_scaled, y_train_t)
    
    y_pred_t = clf_t.predict(X_test_scaled)
    f1_t = f1_score(y_test_t, y_pred_t)
    
    results.append({
        'thresh': thresh,
        'pos_rate': pos_rate,
        'f1': f1_t
    })
    print(f'Thresh={thresh}: pos_rate={100*pos_rate:.1f}%, F1={f1_t:.4f}')

# Find best threshold
best = max(results, key=lambda x: x['f1'])
print(f'\nBest: thresh={best["thresh"]}, F1={best["f1"]:.4f}')

In [ ]:
# === SAVE BEST MODEL ===
import pickle

best_thresh = best['thresh']
y_best = (np.array([s['rms_max'] for s in all_samples]) > best_thresh).astype(int)
y_train_b = y_best[train_mask]

clf_best = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_best.fit(X_train_scaled, y_train_b)

with open(f'{OUTPUT_DIR}/energy_model.pkl', 'wb') as f:
    pickle.dump({
        'model': clf_best,
        'scaler': scaler,
        'threshold': best_thresh,
        'n_samples': len(X),
        'positive_rate': float(y_best.mean())
    }, f)

print(f'Model saved to {OUTPUT_DIR}/energy_model.pkl')
print(f'Dataset: {len(X)} samples, {100*y_best.mean():.1f}% positive')